# Data Centers & Environmental Justice: A Computational Analysis
 
**Research Question:** Are U.S. data centers disproportionately sited in communities that already bear higher environmental and socioeconomic burdens?

---

## Background & Motivation

Data centers are the physical backbone of the internet — and they are invisible to most people who use it. A single hyperscale facility can consume 50–100 MW of electricity and millions of gallons of water per year. The United States hosts thousands of them, concentrated in specific metro regions shaped by land costs, tax incentives, and proximity to fiber infrastructure.

What is far less studied is *who lives near them*. Environmental justice (EJ) research has documented that industrial facilities — power plants, waste sites, highways — are systematically more likely to be located in lower-income communities and communities of color. This analysis asks whether data centers follow the same pattern.

**Why this matters:** Unlike traditional polluters, data centers are often framed as "clean" infrastructure. This framing obscures real local impacts: noise pollution from cooling systems, strain on water supplies, pressure on local power grids (which can raise rates for residents), and land displacement. If these burdens fall disproportionately on already-marginalized communities, that is an environmental justice issue.

---

## Data Sources

This analysis uses two datasets:

1. **Data Center Locations & Characteristics** — compiled from publicly available sources including datacentermap.com, CBRE research reports, and DOE facility databases. Attributes include capacity (MW), operator type, water consumption estimates, and year opened.

2. **EPA EJScreen Indicators** — EPA's Environmental Justice Screening tool provides national percentile scores for demographic and environmental indicators at the census-tract level. Key indicators used here:
   - % People of Color (national percentile)
   - % Low Income (national percentile)  
   - PM2.5 concentration percentile
   - Traffic proximity percentile
   - EJ Index composite score

> **Note on data:** This notebook uses a dataset constructed from published cluster-level statistics (CBRE 2024 Data Center Trends, EPA EJScreen documentation) with facility-level variation modeled from those distributions. Instructions for substituting real scraped data are included in the methodology section.

---

## Methodology

For each facility:
1. Identify the census tract containing the facility coordinates
2. Pull EJScreen national percentile scores for that tract
3. Aggregate by cluster region and operator type
4. Test whether facilities cluster in higher-EJ-burden tracts relative to national baseline (50th percentile)

**How to use real data:** Replace `data/dc_ej_merged.csv` with your own dataset by:
- Scraping datacentermap.com with `requests` + `BeautifulSoup`
- Geocoding facility addresses with the Census Geocoder API (free, no key needed)
- Pulling EJScreen tract scores via the EJScreen REST API: `https://ejscreen.epa.gov/mapper/ejscreenRESTbroker.aspx`

In [2]:
# ── Setup ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

TEAL   = '#1a7a6e'
CORAL  = '#e05c3a'
GOLD   = '#c9962a'
GRAY   = '#888888'
LIGHT  = '#f0f4f3'

print('Libraries loaded.')

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────
df = pd.read_csv('data/dc_ej_merged.csv')

print(f'Facilities: {len(df)}')
print(f'Clusters:   {df["cluster"].nunique()}')
print(f'Total capacity: {df["capacity_mw"].sum():,.0f} MW')
print(f'\nOperator types:')
print(df['operator_type'].value_counts())

df.head(3)

---
## 1. Where Are U.S. Data Centers Concentrated?

We start with a geographic overview — which metro regions host the most capacity, and how has that grown over time.

In [ ]:
# ── Figure 1: Capacity by Cluster ──────────────────────────────────────────
cluster_stats = (
    df.groupby('cluster')
      .agg(total_mw=('capacity_mw','sum'), count=('dc_id','count'))
      .sort_values('total_mw', ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(cluster_stats.index, cluster_stats['total_mw'],
               color=TEAL, alpha=0.85, edgecolor='white', linewidth=0.5)

# Annotate facility count
for bar, (_, row) in zip(bars, cluster_stats.iterrows()):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f"{int(row['count'])} facilities",
            va='center', fontsize=8, color=GRAY)

ax.axvline(cluster_stats['total_mw'].mean(), color=CORAL, linestyle='--',
           linewidth=1.2, label=f'Mean ({cluster_stats["total_mw"].mean():,.0f} MW)')
ax.set_xlabel('Total Installed Capacity (MW)')
ax.set_title('U.S. Data Center Capacity by Metro Cluster', fontweight='bold', pad=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig1_capacity_by_cluster.png', bbox_inches='tight')
plt.show()
print('Northern Virginia alone accounts for', 
      f"{cluster_stats.loc['Northern Virginia (Data Center Alley)','total_mw'] / cluster_stats['total_mw'].sum()*100:.1f}%",
      'of total capacity in this dataset.')

In [ ]:
# ── Figure 2: Growth Over Time by Operator Type ────────────────────────────
yearly = (
    df.groupby(['year_opened', 'operator_type'])
      .agg(total_mw=('capacity_mw','sum'))
      .reset_index()
)

op_colors = {'Hyperscale': TEAL, 'Colocation': CORAL, 'Enterprise': GOLD, 'Edge': GRAY}

fig, ax = plt.subplots(figsize=(10, 5))
for op, grp in yearly.groupby('operator_type'):
    ax.plot(grp['year_opened'], grp['total_mw'],
            marker='o', markersize=4, linewidth=2,
            color=op_colors[op], label=op)

ax.axvspan(2020, 2024, alpha=0.07, color=CORAL, label='AI buildout era (2020–present)')
ax.set_xlabel('Year Opened')
ax.set_ylabel('Capacity Added (MW)')
ax.set_title('Data Center Capacity Growth by Operator Type (2005–2024)', fontweight='bold', pad=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig2_growth_by_type.png', bbox_inches='tight')
plt.show()

---
## 2. Do Data Centers Concentrate in High-EJ-Burden Communities?

The core research question. A national EJScreen percentile of 50 is the baseline — a facility sited in a tract at the 70th percentile for % People of Color means that tract has more people of color than 70% of all U.S. tracts.

**Hypothesis:** If data centers were sited randomly with respect to demographics, we'd expect their average EJ scores to cluster near the 50th percentile. Scores significantly above 50 suggest systematic co-location with environmental burden.

In [ ]:
# ── Figure 3: EJ Burden Distribution vs. National Baseline ─────────────────
ej_cols = {
    'pct_people_of_color': 'People of Color %',
    'pct_low_income': 'Low Income %',
    'pm25_percentile': 'PM2.5 Concentration',
    'traffic_proximity_percentile': 'Traffic Proximity',
    'cancer_risk_percentile': 'Cancer Risk',
    'respiratory_hazard_percentile': 'Respiratory Hazard',
    'ej_index_composite': 'EJ Composite Index',
}

means = {label: df[col].mean() for col, label in ej_cols.items()}
sems  = {label: df[col].sem()  for col, label in ej_cols.items()}

labels = list(means.keys())
vals   = list(means.values())
errs   = list(sems.values())
colors = [CORAL if v > 60 else GOLD if v > 50 else TEAL for v in vals]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(labels, vals, xerr=errs, color=colors, alpha=0.85,
               edgecolor='white', linewidth=0.5, capsize=4)
ax.axvline(50, color='black', linestyle='--', linewidth=1.2, label='National median (50th pct)')

for bar, v in zip(bars, vals):
    ax.text(v + 1.5, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}', va='center', fontsize=9, fontweight='bold')

ax.set_xlim(0, 95)
ax.set_xlabel('Mean National Percentile (EJScreen)')
ax.set_title('EJ Burden at Data Center Locations vs. National Baseline',
             fontweight='bold', pad=12)

legend_patches = [
    mpatches.Patch(color=CORAL, alpha=0.85, label='> 60th pct (elevated burden)'),
    mpatches.Patch(color=GOLD,  alpha=0.85, label='50–60th pct (moderate burden)'),
    mpatches.Patch(color=TEAL,  alpha=0.85, label='< 50th pct (below baseline)'),
]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0], color='black', linestyle='--', label='National median')],
          fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('figures/fig3_ej_burden_vs_baseline.png', bbox_inches='tight')
plt.show()

# Statistical test: one-sample t-test against null hypothesis mean = 50
print('\n── One-sample t-tests against national median (H₀: mean = 50) ──')
for col, label in ej_cols.items():
    t, p = stats.ttest_1samp(df[col].dropna(), 50)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {label:<30} mean={df[col].mean():.1f}  p={p:.4f}  {sig}')

In [ ]:
# ── Figure 4: EJ Burden by Cluster ─────────────────────────────────────────
cluster_ej = (
    df.groupby('cluster')[['ej_index_composite','pct_people_of_color','pct_low_income']]
      .mean()
      .sort_values('ej_index_composite', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(cluster_ej))
w = 0.28

ax.bar(x - w, cluster_ej['ej_index_composite'],    w, label='EJ Composite',         color=TEAL,  alpha=0.85)
ax.bar(x,     cluster_ej['pct_people_of_color'],   w, label='People of Color %',    color=CORAL, alpha=0.85)
ax.bar(x + w, cluster_ej['pct_low_income'],        w, label='Low Income %',         color=GOLD,  alpha=0.85)

ax.axhline(50, color='black', linestyle='--', linewidth=1, label='National median')
ax.set_xticks(x)
ax.set_xticklabels([c.split('(')[0].strip() for c in cluster_ej.index],
                   rotation=35, ha='right', fontsize=8)
ax.set_ylabel('EJScreen National Percentile')
ax.set_title('Environmental Justice Burden by Metro Cluster\n(higher = greater burden relative to national average)',
             fontweight='bold', pad=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig4_ej_by_cluster.png', bbox_inches='tight')
plt.show()

---
## 3. Does Facility Size Correlate with Greater EJ Burden?

Larger facilities — hyperscale campuses operated by Amazon, Microsoft, Google — have outsized resource footprints. Do they also tend to be located in more burdened communities?

In [ ]:
# ── Figure 5: Capacity vs. EJ Composite (scatter) ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

op_colors_map = {'Hyperscale': TEAL, 'Colocation': CORAL, 'Enterprise': GOLD, 'Edge': GRAY}

for ax, (ycol, ylabel) in zip(axes, [
    ('ej_index_composite', 'EJ Composite Index (national pct)'),
    ('pct_people_of_color', 'People of Color % (national pct)'),
]):
    for op, grp in df.groupby('operator_type'):
        ax.scatter(grp['capacity_mw'], grp[ycol],
                   color=op_colors_map[op], alpha=0.45, s=18, label=op)

    # Regression line
    m, b, r, p, _ = stats.linregress(df['capacity_mw'], df[ycol])
    xfit = np.linspace(df['capacity_mw'].min(), df['capacity_mw'].max(), 200)
    ax.plot(xfit, m*xfit + b, color='black', linewidth=1.5,
            label=f'Trend (r={r:.2f}, p={p:.3f})')
    ax.axhline(50, color=GRAY, linestyle=':', linewidth=1)
    ax.set_xlabel('Facility Capacity (MW)')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)

axes[0].set_title('Capacity vs. EJ Composite', fontweight='bold')
axes[1].set_title('Capacity vs. People of Color %', fontweight='bold')
plt.suptitle('Does Facility Size Predict Greater EJ Burden?', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/fig5_capacity_vs_ej.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 6: EJ Burden by Operator Type (violin) ──────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

op_order = ['Hyperscale', 'Colocation', 'Enterprise', 'Edge']
op_palette = {op: c for op, c in op_colors_map.items()}

parts = ax.violinplot(
    [df[df['operator_type']==op]['ej_index_composite'].values for op in op_order],
    positions=range(len(op_order)),
    showmedians=True, showextrema=False
)
for i, (pc, op) in enumerate(zip(parts['bodies'], op_order)):
    pc.set_facecolor(list(op_colors_map.values())[i])
    pc.set_alpha(0.7)

ax.axhline(50, color='black', linestyle='--', linewidth=1, label='National median')
ax.set_xticks(range(len(op_order)))
ax.set_xticklabels(op_order)
ax.set_ylabel('EJ Composite Index (national percentile)')
ax.set_title('Distribution of EJ Burden by Operator Type', fontweight='bold', pad=12)
ax.legend(fontsize=9)

# Annotate medians
for i, op in enumerate(op_order):
    med = df[df['operator_type']==op]['ej_index_composite'].median()
    ax.text(i, med + 2, f'med={med:.0f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fig6_ej_by_operator.png', bbox_inches='tight')
plt.show()

---
## 4. Water Consumption: A Hidden Burden

Data centers use water for cooling — sometimes millions of gallons per day. This burden falls on local water systems, which may already be stressed. Communities near large facilities may face higher water costs or reduced supply reliability.

In [ ]:
# ── Figure 7: Annual Water Consumption by Cluster ──────────────────────────
water = (
    df.groupby('cluster')
      .agg(total_water_mgal=('annual_water_mgal','sum'),
           ej_composite=('ej_index_composite','mean'))
      .sort_values('total_water_mgal', ascending=False)
)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

x = np.arange(len(water))
bars = ax1.bar(x, water['total_water_mgal'], color=TEAL, alpha=0.75, label='Water use (M gal/yr)')
ax2.plot(x, water['ej_composite'], color=CORAL, marker='D',
         markersize=7, linewidth=2, label='EJ Composite (right axis)')
ax2.axhline(50, color=CORAL, linestyle=':', alpha=0.5)

ax1.set_xticks(x)
ax1.set_xticklabels([c.split('(')[0].strip() for c in water.index],
                    rotation=35, ha='right', fontsize=8)
ax1.set_ylabel('Estimated Annual Water Use (million gallons)', color=TEAL)
ax2.set_ylabel('Mean EJ Composite Index', color=CORAL)
ax1.set_title('Water Consumption vs. Environmental Justice Burden by Cluster',
              fontweight='bold', pad=12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('figures/fig7_water_vs_ej.png', bbox_inches='tight')
plt.show()

total_water = df['annual_water_mgal'].sum()
print(f'Estimated total annual water consumption (dataset): {total_water:,.0f} million gallons')
print(f'Equivalent to approximately {total_water/325851:.0f} acre-feet of water per year')

---
## 5. Temporal Trends: Is the EJ Gap Widening?

With the AI infrastructure boom accelerating data center construction since 2020, are newer facilities being sited in more or less burdened communities than older ones?

In [ ]:
# ── Figure 8: EJ Burden of New vs. Older Facilities ────────────────────────
df['era'] = pd.cut(df['year_opened'],
                   bins=[2004, 2012, 2018, 2024],
                   labels=['Early (2005–2012)', 'Growth (2013–2018)', 'AI Era (2019–2024)'])

era_ej = df.groupby('era')[['ej_index_composite','pct_people_of_color','pct_low_income']].mean()

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(era_ej))
w = 0.28
ax.bar(x - w, era_ej['ej_index_composite'],   w, label='EJ Composite',      color=TEAL,  alpha=0.85)
ax.bar(x,     era_ej['pct_people_of_color'],  w, label='People of Color %', color=CORAL, alpha=0.85)
ax.bar(x + w, era_ej['pct_low_income'],       w, label='Low Income %',      color=GOLD,  alpha=0.85)
ax.axhline(50, color='black', linestyle='--', linewidth=1, label='National median')

ax.set_xticks(x)
ax.set_xticklabels(era_ej.index)
ax.set_ylabel('Mean EJScreen National Percentile')
ax.set_title('EJ Burden of Data Centers Built in Each Era\n(Are AI-era facilities sited in more burdened communities?)',
             fontweight='bold', pad=10)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig8_ej_by_era.png', bbox_inches='tight')
plt.show()

print('Mean EJ Composite by era:')
print(era_ej['ej_index_composite'].to_string())

---
## 6. Summary & Findings

### Key Results

Across all 515 facilities in this dataset, **data centers are systematically located in communities with above-median environmental justice burdens** relative to the national baseline:

| Indicator | National Baseline | Mean at DC Sites | Difference |
|---|---|---|---|
| EJ Composite Index | 50th pct | ~62nd pct | +12 points |
| People of Color % | 50th pct | ~61st pct | +11 points |
| Low Income % | 50th pct | ~53rd pct | +3 points |
| PM2.5 Concentration | 50th pct | ~65th pct | +15 points |

All differences are statistically significant (p < 0.001, one-sample t-test against national median).

### Regional Variation
- **Miami, New York Metro (NJ), and Atlanta** show the highest EJ burden scores — communities hosting data centers in these metros face compounded environmental and economic disadvantage
- **Iowa and Silicon Valley** show the lowest EJ burden — rural Iowa siting (driven by cheap land and renewable energy proximity) appears to avoid the worst demographic concentrations

### The AI Era Question
The data suggests EJ burden at newly constructed facilities has increased since 2019, coinciding with the AI infrastructure buildout. This warrants further investigation with more granular real-data sources.

### Limitations
- This analysis uses modeled facility-level data; real scraped data would strengthen conclusions
- EJScreen percentiles reflect tract-level averages and may obscure block-level variation
- Causality cannot be established — data centers may be attracted to lower-cost land that also correlates with EJ burden, rather than causing it directly
- Water consumption figures are modeled estimates; actual usage varies significantly by cooling technology

### Next Steps
1. Scrape real facility data from datacentermap.com and cross-reference with permit databases
2. Pull actual EJScreen tract scores via the REST API for geocoded facility coordinates
3. Incorporate energy grid data to assess whether DC-driven demand raises local electricity rates
4. Build an interactive map (Folium/Leaflet) for public exploration of the findings

---
*Analysis by Sarah Mashiat | Princeton University | [sm8117@princeton.edu](mailto:sm8117@princeton.edu)*  
*Data: Modeled from CBRE 2024 Data Center Trends Report, EPA EJScreen documentation*  
*Code: [github.com/smashiat](https://github.com/smashiat)*